In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [3]:
len(documents)

72

In [4]:
from minsearch import Index

index = Index(
    keyword_fields=["filename"],
    text_fields=["content"]
)

index.fit(documents)

In [5]:
search_documents = index.search("How does the agentic loop keep calling the model until it stops?")
search_documents[0]

{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry 

In [6]:
from rag_helper import RAGBase
from openai import OpenAI

openai_client = OpenAI()

assistant = RAGBase(
    index = index,
    llm_client = openai_client
)

In [7]:
answer = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(answer)

Response(id='resp_0124bdb0581b6e01006a3a825b937481a089b01cf452e1f929', created_at=1782219355.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseOutputMessage(id='msg_0124bdb0581b6e01006a3a825c1ebc81a09657809f34916f48', content=[ResponseOutputText(annotations=[], text='The agentic loop keeps calling the model with a `while True` loop and stops only when the model returns **no function calls** in that turn.\n\nIn the lesson, the logic is:\n\n1. Call the model with the current `messages`.\n2. Check the response:\n   - if it contains a `function_call`, run the tool, append the tool output to `messages`, and loop again;\n   - if it contains only a final `message`, set `has_function_calls = False`.\n3. Break the loop when `has_function_calls == False`.\n\nSo the model “stops” by choosing to answer directly instead of asking for another tool call. The code just keeps repeating until that happens.\n\nDo yo

In [8]:
usage = answer.usage
print(usage)

usage.input_tokens



ResponseUsage(input_tokens=7134, input_tokens_details=InputTokensDetails(cached_tokens=6912), output_tokens=167, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7301)


7134

In [9]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

print(len(chunks))

295


In [10]:
from minsearch import Index

index = Index(
    keyword_fields=["filename"],
    text_fields=["content"]
)

index.fit(chunks)

In [11]:
from rag_helper import RAGBase
from openai import OpenAI

openai_client = OpenAI()

assistant = RAGBase(
    index = index,
    llm_client = openai_client
)

In [12]:
answer = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(answer)

Response(id='resp_0ee7a816447f07ab006a3a825e5db0819d9d0a3824f85db6f6', created_at=1782219358.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseOutputMessage(id='msg_0ee7a816447f07ab006a3a825ef2e4819db002503f2ec24990', content=[ResponseOutputText(annotations=[], text='The loop keeps calling the model by using a `while True` loop and checking whether the model returned any `function_call` items.\n\nHow it works:\n- The model is called with the current `messages`.\n- If the response contains a `function_call`, your code runs the tool, appends the tool result back into `messages`, and sets `has_function_calls = True`.\n- If the response contains only a final `message`, then no tool was needed.\n- After each turn, the code checks:\n  ```python\n  if has_function_calls == False:\n      break\n  ```\n  So the loop stops only when the model gives an answer with no function calls.\n\nIn other words, the mo

In [13]:
usage = answer.usage
print(usage)

usage.input_tokens

ResponseUsage(input_tokens=2319, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=187, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=2506)


2319

In [14]:
def search(query: str) -> dict[str]:
    """
    Search the course pages chunks for matching entries relevant to the question.
    """
    
    return index.search(
        query,
        num_results = 5,
        boost_dict={"content": 3.0,}
    )

In [15]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [16]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [17]:
instructions = """
You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.
""".strip()

In [18]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools = agent_tools,
    developer_prompt = instructions,
    chat_interface = chat_interface,
    llm_client = OpenAIClient(model="gpt-5.4-mini")
)

In [19]:
result = runner.loop(
    prompt = "How does the agentic loop work, and how is it different from plain RAG?",
    callback = callback
)

-> Response received


-> Response received
